## 03c_outpatient_gold — RCM Intelligence Platform
### Purpose
Builds Gold aggregations for outpatient claims.
Mirrors 03b_kpi_aggregations but for outpatient data.
Combined with inpatient Gold to power the full platform.

### What this does
1. Reads outpatient_claims_enriched from Silver
2. Builds outpatient KPIs by state
3. Builds outpatient KPIs by APC code
4. Builds combined inpatient + outpatient hospital scorecard
5. Writes all tables to rcm_gold schema

### Outputs
- rcm_platform.rcm_gold.outpatient_kpi_by_state
- rcm_platform.rcm_gold.outpatient_kpi_by_apc
- rcm_platform.rcm_gold.combined_hospital_scorecard

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Gold |
| Run order  | 7b — after 03b and 02c |
| Depends on | 02c_outpatient_transform, 03b_kpi_aggregations |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
import builtins
round = builtins.round

from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, expr,
    round as spark_round,
    sum as spark_sum,
    avg as spark_avg,
    count, countDistinct,
    percentile_approx
)

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "03c_outpatient_gold"

TBL_OUT_ENRICHED          = f"{SILVER_DB}.outpatient_claims_enriched"
TBL_GOLD_OUT_STATE        = f"{GOLD_DB}.outpatient_kpi_by_state"
TBL_GOLD_OUT_APC          = f"{GOLD_DB}.outpatient_kpi_by_apc"
TBL_GOLD_COMBINED_SCORECARD = f"{GOLD_DB}.combined_hospital_scorecard"

print(f"Batch ID : {BATCH_ID}")
print(f"Outputs  :")
print(f"  {TBL_GOLD_OUT_STATE}")
print(f"  {TBL_GOLD_OUT_APC}")
print(f"  {TBL_GOLD_COMBINED_SCORECARD}")

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
import builtins
round = builtins.round

from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, expr,
    round as spark_round,
    sum as spark_sum,
    avg as spark_avg,
    count, countDistinct,
    percentile_approx
)

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "03c_outpatient_gold"

TBL_OUT_ENRICHED          = f"{SILVER_DB}.outpatient_claims_enriched"
TBL_GOLD_OUT_STATE        = f"{GOLD_DB}.outpatient_kpi_by_state"
TBL_GOLD_OUT_APC          = f"{GOLD_DB}.outpatient_kpi_by_apc"
TBL_GOLD_COMBINED_SCORECARD = f"{GOLD_DB}.combined_hospital_scorecard"

print(f"Batch ID : {BATCH_ID}")
print(f"Outputs  :")
print(f"  {TBL_GOLD_OUT_STATE}")
print(f"  {TBL_GOLD_OUT_APC}")
print(f"  {TBL_GOLD_COMBINED_SCORECARD}")

In [0]:
# ============================================================
# STEP 1 — READ SILVER OUTPATIENT
# ============================================================

print("=" * 55)
print("  READING SILVER OUTPATIENT")
print("=" * 55)

df_out = spark.table(TBL_OUT_ENRICHED)
df_inp = spark.table(TBL_FACT_CLAIMS)
df_scorecard = spark.table(TBL_GOLD_SCORECARD)

print(f"Outpatient rows : {df_out.count():,}")
print(f"Inpatient rows  : {df_inp.count():,}")
print(f"Scorecard rows  : {df_scorecard.count():,}")

In [0]:
# ============================================================
# STEP 2 — OUTPATIENT KPI BY STATE
# ============================================================

print("=" * 55)
print("  BUILDING OUTPATIENT KPI BY STATE")
print("=" * 55)

outpatient_state = spark.sql(f"""
    SELECT
        provider_state,
        COUNT(DISTINCT provider_id)                        AS hospital_count,
        COUNT(DISTINCT apc_code)                           AS unique_apc_codes,
        SUM(beneficiary_count)                             AS total_beneficiaries,
        ROUND(AVG(charge_to_payment_ratio), 2)             AS avg_ctp_ratio,
        ROUND(AVG(revenue_gap_pct), 2)                     AS avg_revenue_gap_pct,
        ROUND(SUM(total_revenue_gap) / 1e9, 2)             AS total_revenue_gap_billions,
        ROUND(SUM(underpayment_flag) / COUNT(*) * 100, 2)  AS underpayment_rate_pct,
        '{BATCH_DATE}'                                     AS _batch_date
    FROM {TBL_OUT_ENRICHED}
    GROUP BY provider_state
    ORDER BY total_revenue_gap_billions DESC
""")

outpatient_state.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_OUT_STATE)

state_rows = spark.table(TBL_GOLD_OUT_STATE).count()
print(f"Outpatient KPI by state : {state_rows} rows written")

display(spark.table(TBL_GOLD_OUT_STATE).limit(10))

In [0]:
# ============================================================
# STEP 3 — OUTPATIENT KPI BY APC CODE
# ============================================================

print("=" * 55)
print("  BUILDING OUTPATIENT KPI BY APC")
print("=" * 55)

outpatient_apc = spark.sql(f"""
    SELECT
        apc_code,
        apc_description,
        COUNT(DISTINCT provider_id)                        AS hospital_count,
        SUM(beneficiary_count)                             AS total_beneficiaries,
        ROUND(AVG(charge_to_payment_ratio), 2)             AS avg_ctp_ratio,
        ROUND(AVG(revenue_gap_pct), 2)                     AS avg_revenue_gap_pct,
        ROUND(AVG(avg_submitted_charge), 2)                AS avg_charge,
        ROUND(AVG(avg_medicare_payment), 2)                AS avg_medicare_payment,
        ROUND(SUM(total_revenue_gap) / 1e9, 2)             AS total_revenue_gap_billions,
        ROUND(SUM(underpayment_flag) / COUNT(*) * 100, 2)  AS underpayment_rate_pct,
        '{BATCH_DATE}'                                     AS _batch_date
    FROM {TBL_OUT_ENRICHED}
    GROUP BY apc_code, apc_description
    ORDER BY total_revenue_gap_billions DESC
""")

outpatient_apc.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_OUT_APC)

apc_rows = spark.table(TBL_GOLD_OUT_APC).count()
print(f"Outpatient KPI by APC : {apc_rows} rows written")

display(spark.table(TBL_GOLD_OUT_APC).limit(10))

In [0]:
# ============================================================
# STEP 4 — COMBINED INPATIENT + OUTPATIENT HOSPITAL SCORECARD
# This is the killer table — full picture per hospital
# ============================================================

print("=" * 55)
print("  BUILDING COMBINED HOSPITAL SCORECARD")
print("=" * 55)

# Outpatient aggregated by hospital
outpatient_by_hospital = spark.sql(f"""
    SELECT
        provider_id,
        COUNT(DISTINCT apc_code)                           AS outpatient_apc_count,
        SUM(beneficiary_count)                             AS outpatient_beneficiaries,
        ROUND(AVG(charge_to_payment_ratio), 2)             AS outpatient_avg_ctp,
        ROUND(AVG(revenue_gap_pct), 2)                     AS outpatient_avg_gap_pct,
        ROUND(SUM(total_revenue_gap) / 1e6, 2)             AS outpatient_revenue_gap_millions,
        ROUND(SUM(underpayment_flag) / COUNT(*) * 100, 2)  AS outpatient_underpayment_rate
    FROM {TBL_OUT_ENRICHED}
    GROUP BY provider_id
""")

outpatient_by_hospital.createOrReplaceTempView("outpatient_by_hospital")

# Join with existing inpatient scorecard
combined = spark.sql(f"""
    SELECT
        s.provider_id,
        s.provider_name,
        s.provider_state,
        s.provider_city,
        s.hospital_type,
        s.hospital_ownership,
        s.emergency_services,

        -- Inpatient metrics
        s.total_discharges                                 AS inpatient_discharges,
        s.avg_ctp_ratio                                    AS inpatient_avg_ctp,
        s.avg_revenue_gap_pct                              AS inpatient_avg_gap_pct,
        s.underpayment_rate_pct                            AS inpatient_underpayment_rate,
        ROUND(s.total_revenue_gap / 1e6, 2)               AS inpatient_revenue_gap_millions,

        -- Outpatient metrics
        COALESCE(o.outpatient_beneficiaries, 0)            AS outpatient_beneficiaries,
        COALESCE(o.outpatient_avg_ctp, 0)                  AS outpatient_avg_ctp,
        COALESCE(o.outpatient_avg_gap_pct, 0)              AS outpatient_avg_gap_pct,
        COALESCE(o.outpatient_underpayment_rate, 0)        AS outpatient_underpayment_rate,
        COALESCE(o.outpatient_revenue_gap_millions, 0)     AS outpatient_revenue_gap_millions,
        COALESCE(o.outpatient_apc_count, 0)                AS outpatient_apc_count,

        -- Combined metrics
        ROUND(
            (COALESCE(s.total_revenue_gap, 0) +
             COALESCE(o.outpatient_revenue_gap_millions * 1e6, 0)) / 1e9
        , 2)                                               AS combined_revenue_gap_billions,

        ROUND(
            (COALESCE(s.avg_ctp_ratio, 0) +
             COALESCE(o.outpatient_avg_ctp, 0)) / 2
        , 2)                                               AS combined_avg_ctp,

        -- Existing RCM scores
        s.rcm_health_score,
        s.rcm_health_grade,
        s.drg_count,
        '{BATCH_DATE}'                                     AS _batch_date

    FROM {TBL_GOLD_SCORECARD} s
    LEFT JOIN outpatient_by_hospital o
        ON s.provider_id = o.provider_id
    ORDER BY combined_revenue_gap_billions DESC
""")

combined.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_COMBINED_SCORECARD)

combined_rows = spark.table(TBL_GOLD_COMBINED_SCORECARD).count()
print(f"Combined scorecard rows : {combined_rows:,}")

print("\nTop 10 hospitals by combined revenue gap:")
display(
    spark.table(TBL_GOLD_COMBINED_SCORECARD)
        .select(
            "provider_name",
            "provider_state",
            "inpatient_revenue_gap_millions",
            "outpatient_revenue_gap_millions",
            "combined_revenue_gap_billions",
            "rcm_health_grade"
        )
        .limit(10)
)

In [0]:
# ============================================================
# STEP 5 — PLATFORM SUMMARY
# ============================================================

print("=" * 55)
print("  FULL PLATFORM SUMMARY")
print("=" * 55)

spark.sql(f"""
    SELECT
        COUNT(*)                                           AS total_hospitals,
        ROUND(SUM(combined_revenue_gap_billions), 2)       AS total_combined_gap_billions,
        ROUND(AVG(combined_avg_ctp), 2)                    AS avg_combined_ctp,
        ROUND(AVG(inpatient_revenue_gap_millions), 2)      AS avg_inpatient_gap_m,
        ROUND(AVG(outpatient_revenue_gap_millions), 2)     AS avg_outpatient_gap_m,
        SUM(CASE WHEN rcm_health_grade = 'A — Excellent'
            THEN 1 ELSE 0 END)                             AS grade_a,
        SUM(CASE WHEN rcm_health_grade = 'F — Critical'
            THEN 1 ELSE 0 END)                             AS grade_f
    FROM {TBL_GOLD_COMBINED_SCORECARD}
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 6 — LOG TO AUDIT
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "gold",
    status           = "SUCCESS",
    rows_read        = df_out.count(),
    rows_written     = combined_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"outpatient KPI by state: {state_rows} rows | "
        f"outpatient KPI by APC: {apc_rows} rows | "
        f"combined scorecard: {combined_rows:,} hospitals"
    )
)